-   Explore columns
-   Figure out metric to quantify "safety"
    -   Crime per 1000: 1000 * (total crime / population)
    -   Weighted crime (weight violent crimes higher than property theft)
-   Sort table by this metric
-   Inspect "Delaware County Sheriff's Office"
-   Compare it with other Ohio counties

## Libraries

In [1]:
import pandas as pd
import sys, os
config_dir = os.path.abspath(os.path.join(os.getcwd(), '..\\config'))
sys.path.append(config_dir)
from ohio_data import ohio_counties, ohio_cities

## Data Ingestion

In [2]:
table = pd.read_excel(r'S:\1. Repositories\ohio-crime-scrapper\data\downloads\ohio_crime_report.ods',
                      engine='odf')
#drop empty column
table.drop(columns=['Unnamed: 11'], inplace = True)

## Feature Engineering

In [6]:
#rename to county level pop since we will add city level as well
table.rename(columns={'Population': 'County Population'}, 
             inplace = True)
#report is pulled from 2022
table['Year'] = '2022'
#total crime
table['Total Crime'] = table['ViolentCrimeTotal'] + table['PropertyCrimeTotal']
#crime per 1000
table['County Crime per 1000'] = (table['Total Crime'] / table['County Population']) * 1000
#creat county column 
table['County'] = table['AgencyName'].str.split('Sheriff').str[0].str.strip()
#add city level data
table["City"] = table["County"].map(ohio_counties)
table = table.explode("City", ignore_index=True)
#add city population data
table["City Population"] = table["City"].map(ohio_cities)
#city level crime rate
table['City Crime per 1000'] = (table['Total Crime'] / table['City Population']) * 1000



In [12]:
features = ['County', 'County Population', 'County Crime per 1000', 'City', 'City Population', 'City Crime per 1000']
table[table['City'].isin(["Cincinnati", "Columbus"])][features]

,County,County Population,County Crime per 1000,City,City Population,City Crime per 1000
297,Franklin County,55803,5.644858,Columbus,907971.0,0.346927
305,Franklin County,55803,5.644858,Columbus,907971.0,0.346927
313,Franklin County,55803,5.644858,Columbus,907971.0,0.346927
321,Franklin County,55803,5.644858,Columbus,907971.0,0.346927
329,Franklin County,55803,5.644858,Columbus,907971.0,0.346927
337,Franklin County,55803,5.644858,Columbus,907971.0,0.346927
345,Franklin County,55803,5.644858,Columbus,907971.0,0.346927
353,Franklin County,55803,5.644858,Columbus,907971.0,0.346927
411,Hamilton County,118395,8.437856,Cincinnati,NaN,NaN
417,Hamilton County,118395,8.437856,Cincinnati,NaN,NaN


In [ ]:
#split agency name by Sheriff's to gert county
#do a lookup to figure out location